In [1]:
user_id = "tp99965073"
domain = "AXISB.COM"
app_name = "Fraud_&_AML"
yarn_queue = "root.DEV.AML_FRAUD_CS"

EXECUTOR_MEMORY=16
EXECUTOR_CORE=3

INITIAL_EXECUTORS=8
MAX_EXECUTORS=20     # 16x4 = 64 cores; lower this to your approved cap if smaller (YARN grants only what is free)
MIN_EXECUTORS=2

In [2]:
import sys
from pyspark.sql import SparkSession, SQLContext
from pyspark import SparkContext, SparkConf
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark.sql.types as T
from datetime import datetime, date,timedelta
from dateutil.relativedelta import relativedelta
import pandas as pd
import time
import numpy as np


from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))


# Construct the keytab path and the principal
keytab_path = f"/home/{user_id}@axisb.com/{user_id}.keytab"
principal = f"{user_id}@{domain}"

# Execute the commands
!kdestroy
!kinit -k -t "{keytab_path}" {principal}
try:
    spark.stop()
    print("Existing Spark Session Killed")
except:
    print("")
print("Creating new spark session since: ", datetime.now())
spark = SparkSession\
    .builder\
    .appName(f'JH_{user_id}_{app_name}')\
    .config('spark.sql.execution.arrow.enabled', "true")\
    .config('spark.master','yarn')\
    .config('spark.deploy-mode', 'cluster')\
    .config("spark.principal",f"{user_id}@AXISB.COM") \
    .config("spark.keytab",f"{user_id}.keytab")\
    .config('spark.dynamicAllocation.enabled','true') \
    .config('spark.dynamicAllocation.minExecutors',MIN_EXECUTORS) \
    .config('spark.dynamicAllocation.initialExecutors',INITIAL_EXECUTORS) \
    .config('spark.dynamicAllocation.maxExecutors',MAX_EXECUTORS) \
    .config('spark.executor.cores',EXECUTOR_CORE) \
    .config('spark.dynamicAllocation.executorAllocationRatio','0.8')\
    .config('spark.executor.memory',f'{EXECUTOR_MEMORY}g')\
    .config('spark.driver.memory','8g')\
    .config('spark.driver.cores','1')\
    .config('spark.executor.memoryOverhead','6g')\
    .config('spark.sql.shuffle.partitions','6000')\
    .config('spark.shuffle.service.enabled','true')\
    .config('spark.network.timeout','600s')\
    .config('spark.shuffle.io.maxRetries','10')\
    .config('spark.shuffle.io.retryWait','15s')\
    .config('spark.stage.maxConsecutiveAttempts','8')\
    .config('spark.maxRemoteBlockSizeFetchToMem','512m')\
    .config("spark.sql.broadcastTimeout", "-1")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "LEGACY")\
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")\
    .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")\
    .config("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")\
    .config("spark.yarn.queue", yarn_queue)\
    .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')\
    .config('spark.hadoop.hive.exec.dynamic.partition.mode', 'nonstrict')\
    .config('spark.sql.parquet.output.committer.class', 'org.apache.parquet.hadoop.ParquetOutputCommitter')\
    .config('spark.sql.sources.commitProtocolClass', 'org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol')\
    .config("spark.jars", "hdfs:///jars/iceberg-spark-runtime-3.3_2.12-1.6.1.jar,/opt/cloudera/parcels/CDH/jars/ojdbc8.jar")\
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')\
    .config('spark.sql.catalog.spark_catalog', 'org.apache.iceberg.spark.SparkSessionCatalog')\
    .config('spark.sql.catalog.spark_catalog.type', 'hive')\
    .config('spark.sql.adaptive.enabled', 'true')\
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')\
    .config('spark.sql.adaptive.skewJoin.enabled', 'true')\
    .config('spark.sql.adaptive.autoBroadcastJoinThreshold','50MB')\
    .config("spark.sql.legacy.allowNonEmptyLocationInCTAS", "true")\
    .enableHiveSupport()\
    .getOrCreate()
sc = SparkContext.getOrCreate()
print("New Spark Session created at : ", datetime.now())

new_log_level = "ERROR"
sc.setLogLevel(new_log_level)
print(f"Logs level set to {new_log_level}")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


Creating new spark session since:  2026-06-25 03:54:41.285247


26/06/25 03:54:42 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/06/25 03:54:42 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/25 03:54:42 WARN  spark.SparkConf: [main]: The configuration key 'spark.maxRemoteBlockSizeFetchToMem' has been deprecated as of Spark 3.0 and may be removed in the future. Please use the new key 'spark.network.maxRemoteBlockSizeFetchToMem' instead.
26/06/25 03:54:43 WARN  spark.SparkConf: [Thread-9]: The configuration key 'spark.maxRemoteBlockSizeFetch

New Spark Session created at :  2026-06-25 03:54:54.666718
Logs level set to ERROR


# Notebook 02 (v2) — Axis CASA dimension + node universe + structural edges

First notebook of the v2 rebuild (full heterogeneous transaction graph). Builds the **dimension /
structural** layer that 02b and 03 use:

* **`axis_account_dim`** — every INTERNAL (Axis) account = `SA_ACCOUNT_DIM.foracid` (savings) UNION
  `CDW_ACCOUNT_DIM.ext_acct_num` (current, `rec_valid_flag='Y'`), with `cust`, `salary_label_code`,
  `casa_type`. This is BOTH the ACCOUNT(internal) node source AND the Axis-flag lookup for 02b.
* **CUSTOMER nodes** + **OWNS_ACCOUNT** edges (cust -> account).
* **SALARY_LABEL nodes** + **HAS_SALARY_LABEL** edges (account -> salary_label_code).
* **`pl_customer_map`** — PL customer -> any-bad flag, first-bad month, loan months (so 03 derives
  per-anchor seeds = historical bad, candidates = window loans).

Reads notebook-01 base (unchanged). Dimension build only (no f_txn_sa). Run session cells first,
then top to bottom. See `NEW_ARCH_SPEC_V2.md` and `Algorithm_Implementation_Plan.md`.

In [3]:
# ---- PARAMETERS ----
from pyspark.sql import functions as F
from pyspark.sql import Window as W

WRITE_DB = "payment_cards_dev"
EMP      = "TP99965073"

SA_DIM   = "BIU_DM.SA_ACCOUNT_DIM"                       # savings accounts (read-only)
CDW_DIM  = "biu_dm.CDW_ACCOUNT_DIM"                      # current accounts (read-only)
BASE_TBL = f"{WRITE_DB}.{EMP}_pl_graph_base_cohort_v1"   # notebook 01 (unchanged)

# SA_ACCOUNT_DIM columns
SA_ACCT, SA_CUST, SA_SALLBL = "foracid", "cust_id", "salary_label_code"
SA_DELFLG = "delete_flag"                                # 'N' = not deleted (keep closed accounts for tagging recall)
# CDW_ACCOUNT_DIM columns
CDW_ACCT, CDW_CUST, CDW_VALID = "ext_acct_num", "cust_id", "rec_valid_flag"   # rec_valid_flag (NOT rev_valid_flag)
# base (notebook 01) columns — saved customer col is lddf_customer_id (cust_id_finacle is a v1 code alias only)
BASE_CUST, BASE_MONTH, BASE_FRAUD = "lddf_customer_id", "month_period", "final_fraud_flag"

ANCHORS   = ["202509", "202510", "202511"]               # month_period (yyyyMM) study anchors
DUMMY_SAL = ["INSTKNEW", "NOSALARY", "PTYVCIP", "NA", "NULL", "0", "-1"]   # non-salary/dummy label codes

OUT_AXIS   = f"{WRITE_DB}.{EMP}_v2_axis_account_dim"     # INTERNAL account attrs + Axis-tag lookup
OUT_CUST   = f"{WRITE_DB}.{EMP}_v2_nodes_customer"
OUT_SAL    = f"{WRITE_DB}.{EMP}_v2_nodes_salary"
OUT_OWN    = f"{WRITE_DB}.{EMP}_v2_edges_ownership"
OUT_HASLBL = f"{WRITE_DB}.{EMP}_v2_edges_salary_label"
OUT_PLMAP  = f"{WRITE_DB}.{EMP}_v2_pl_customer_map"


def ACC_KEY(c):
    return F.upper(F.trim(F.col(c).cast("string")))      # no zero-padding


def save_overwrite(df, t):
    spark.sql("DROP TABLE IF EXISTS " + t)
    df.write.mode("overwrite").saveAsTable(t)


print("SA :", SA_DIM, "| CDW:", CDW_DIM, "| base:", BASE_TBL)
print("outputs:", OUT_AXIS, OUT_CUST, OUT_SAL, OUT_OWN, OUT_HASLBL, OUT_PLMAP)

SA : BIU_DM.SA_ACCOUNT_DIM | CDW: biu_dm.CDW_ACCOUNT_DIM | base: payment_cards_dev.TP99965073_pl_graph_base_cohort_v1
outputs: payment_cards_dev.TP99965073_v2_axis_account_dim payment_cards_dev.TP99965073_v2_nodes_customer payment_cards_dev.TP99965073_v2_nodes_salary payment_cards_dev.TP99965073_v2_edges_ownership payment_cards_dev.TP99965073_v2_edges_salary_label payment_cards_dev.TP99965073_v2_pl_customer_map


## A. Confirm source schemas (defensive)

In [4]:
for t in [SA_DIM, CDW_DIM, BASE_TBL]:
    try:
        print(t, "->", spark.table(t).columns)
    except Exception as e:
        print("ERROR reading", t, ":", e)

Hive Session ID = fc232d9e-4436-4cd2-837a-03d8283b4e95


BIU_DM.SA_ACCOUNT_DIM -> ['account_key', 'acct_anchor_skey', 'cust_anchor_skey', 'acid', 'foracid', 'cust_id', 'acct_name', 'acct_open_date', 'acct_activation_date', 'final_acct_open_date', 'etb_ntb_flag', 'acct_cls_date', 'closure_reason', 'instakit_flag', 'instakit_status', 'schm_type', 'schm_code', 'sol_id', 'product', 'segment', 'product_c', 'segment_c', 'acct_label_code', 'acct_label_desc', 'salary_label_code', 'barcode', 'lc_code', 'lg_code', 'funding_date', 'funding_amt', 'funding_dayend_bal', 'acct_crncy_code', 'acct_status', 'acct_status_date', 'gl_sub_head_code', 'gl_sub_head_code_desc', 'freeze_code', 'freeze_reason_code', 'freeze_remarks', 'freez_date', 'unfreez_date', 'lien_flag', 'nom_available_flag', 'mode_of_oper_code', 'mode_of_oper_code_desc', 'allow_sweeps', 'client_processor', 'client_pro_type', 'is_digital_flag', 'asst_d2c_flag', 'dbat_variant', 'mb_min_txn_fin', 'mb_min_txn_nonfin', 'ib_min_txn_fin', 'ib_min_txn_nonfin', 'is_cheque_issued', 'citi_flag', 'wave', 'd

## B. Axis CASA account dimension  (SA savings UNION CDW current)

In [5]:
sa_sel = (spark.table(SA_DIM)
          .where(F.col(SA_DELFLG) == "N")                # exclude deleted records; keep closed (better tag recall)
          .select(ACC_KEY(SA_ACCT).alias("acct"),
                  ACC_KEY(SA_CUST).alias("cust"),
                  F.col(SA_SALLBL).cast("string").alias("salary_label_code"),
                  F.lit("SA").alias("casa_type")))

cdw_sel = (spark.table(CDW_DIM)
           .where(F.col(CDW_VALID) == "Y")
           .select(ACC_KEY(CDW_ACCT).alias("acct"),
                   ACC_KEY(CDW_CUST).alias("cust"),
                   F.lit(None).cast("string").alias("salary_label_code"),
                   F.lit("CA").alias("casa_type")))

both = sa_sel.unionByName(cdw_sel).where(F.col("acct").isNotNull() & (F.col("acct") != ""))

# one row per account (SA contributes salary_label_code; SA/CA account numbers should not collide)
axis = (both.groupBy("acct").agg(
            F.max("cust").alias("cust"),
            F.max("salary_label_code").alias("salary_label_code"),
            F.min("casa_type").alias("casa_type"))
        .withColumn("node_type", F.lit("ACCOUNT"))
        .withColumn("account_network_type", F.lit("INTERNAL")))

save_overwrite(axis, OUT_AXIS)
axis = spark.table(OUT_AXIS)
print("axis_account_dim rows:", axis.count())
axis.groupBy("casa_type").count().orderBy("casa_type").show()

axis_account_dim rows: 79918733


[Stage 6:======================================================>  (58 + 3) / 61]

+---------+--------+
|casa_type|   count|
+---------+--------+
|       CA| 6762236|
|       SA|73156497|
+---------+--------+



## C. CUSTOMER nodes + OWNS_ACCOUNT edges

In [6]:
cust_nodes = (axis.select(F.col("cust").alias("node_id"))
              .where(F.col("cust").isNotNull() & (F.trim(F.col("cust")) != ""))
              .distinct()
              .withColumn("node_type", F.lit("CUSTOMER")))
save_overwrite(cust_nodes, OUT_CUST)

own = (axis.where(F.col("cust").isNotNull() & (F.trim(F.col("cust")) != ""))
       .select(F.col("cust").alias("src"), F.col("acct").alias("dst"))
       .withColumn("edge_type", F.lit("OWNS_ACCOUNT")))
save_overwrite(own, OUT_OWN)

print("customer nodes :", spark.table(OUT_CUST).count())
print("ownership edges:", spark.table(OUT_OWN).count())

customer nodes : 72727336
ownership edges: 79918733


## D. SALARY_LABEL nodes + HAS_SALARY_LABEL edges

In [7]:
_sal_key = F.upper(F.trim(F.col("salary_label_code").cast("string")))
_sal_ok = (F.col("salary_label_code").isNotNull()
           & (_sal_key != "") & (~_sal_key.isin(*[d.upper() for d in DUMMY_SAL])))

sal_nodes = (axis.where(_sal_ok).select(_sal_key.alias("node_id")).distinct()
             .withColumn("node_type", F.lit("SALARY_LABEL")))
save_overwrite(sal_nodes, OUT_SAL)

haslbl = (axis.where(_sal_ok)
          .select(F.col("acct").alias("src"), _sal_key.alias("dst"))
          .withColumn("edge_type", F.lit("HAS_SALARY_LABEL")))
save_overwrite(haslbl, OUT_HASLBL)

print("salary_label nodes :", spark.table(OUT_SAL).count())
print("has_salary_label edges:", spark.table(OUT_HASLBL).count())

salary_label nodes : 99492
has_salary_label edges: 19673347


## E. PL customer map  (seeds = historical bad; candidates = window loans -> derived in 03)

In [8]:
base = spark.table(BASE_TBL)
_cust_col = next((c for c in [BASE_CUST, "lddf_customer_id", "cust_id_finacle", "cust_id"] if c in base.columns), BASE_CUST)
print("using base customer column:", _cust_col)
plmap = (base.select(ACC_KEY(_cust_col).alias("cust"),
                     F.col(BASE_MONTH).cast("string").alias("month_period"),
                     F.col(BASE_FRAUD).cast("int").alias("final_fraud_flag"))
         .where(F.col("cust").isNotNull() & (F.col("cust") != ""))
         .groupBy("cust")
         .agg(F.max("final_fraud_flag").alias("any_bad"),
              F.min(F.when(F.col("final_fraud_flag") == 1, F.col("month_period"))).alias("first_bad_month"),
              F.collect_set("month_period").alias("loan_months")))
save_overwrite(plmap, OUT_PLMAP)
plmap = spark.table(OUT_PLMAP)
print("pl_customer_map rows:", plmap.count())
plmap.agg(F.sum("any_bad").alias("bad_pl_customers")).show()

# how many PL customers map to an Axis account (sanity vs base)
pl_with_axis = plmap.join(axis.select("cust").distinct(), "cust", "inner").count()
print("PL customers that map to >=1 Axis account:", pl_with_axis)

using base customer column: lddf_customer_id


pl_customer_map rows: 5019501
+----------------+
|bad_pl_customers|
+----------------+
|           19439|
+----------------+



[Stage 42:=======================================================>(60 + 1) / 61]

PL customers that map to >=1 Axis account: 4142131


## Done — QC summary

In [9]:
print("=== Notebook 02 (v2) outputs ===")
for t in [OUT_AXIS, OUT_CUST, OUT_SAL, OUT_OWN, OUT_HASLBL, OUT_PLMAP]:
    print("  %-60s rows: %d" % (t, spark.table(t).count()))
print("NOTEBOOK 02 (v2) COMPLETE — Axis CASA dimension + customer/salary nodes + ownership/salary edges + PL map.")
print("Next: 02b (v2) builds the full transaction graph (all f_txn_sa) and tags INTERNAL/EXTERNAL via axis_account_dim.")
print("Paste back the QC counts (axis_account_dim by casa_type, customer/salary/ownership/salary-label counts,")
print("pl_customer_map rows + bad_pl_customers + PL-with-axis) so 02b can be tuned.")

=== Notebook 02 (v2) outputs ===
  payment_cards_dev.TP99965073_v2_axis_account_dim             rows: 79918733
  payment_cards_dev.TP99965073_v2_nodes_customer               rows: 72727336
  payment_cards_dev.TP99965073_v2_nodes_salary                 rows: 99492
  payment_cards_dev.TP99965073_v2_edges_ownership              rows: 79918733
  payment_cards_dev.TP99965073_v2_edges_salary_label           rows: 19673347
  payment_cards_dev.TP99965073_v2_pl_customer_map              rows: 5019501
NOTEBOOK 02 (v2) COMPLETE — Axis CASA dimension + customer/salary nodes + ownership/salary edges + PL map.
Next: 02b (v2) builds the full transaction graph (all f_txn_sa) and tags INTERNAL/EXTERNAL via axis_account_dim.
Paste back the QC counts (axis_account_dim by casa_type, customer/salary/ownership/salary-label counts,
pl_customer_map rows + bad_pl_customers + PL-with-axis) so 02b can be tuned.
